# 10.1 · Saving and loading models; reproducibility

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/01-saving-and-reproducibility/lesson.ipynb)

Read the [README](README.md) first. This notebook trains and saves the churn bundle, takes it apart, and runs the batch scoring script the way a scheduler would.

## 1. Train and save the bundle

`cm.load_bundle()` in the setup cell trains the model and saves `models/churn_bundle.joblib` the first time; afterwards it just loads the file. This is exactly what `python train_churn.py` does.

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

Model trained 2026-10-28, test AUC 0.741, threshold 0.07


In [2]:
import joblib, sklearn
path = cm.MODEL_PATH
print(path.relative_to(ROOT), f"{path.stat().st_size:,} bytes")
for key, value in bundle.items():
    print(f"  {key:16s} {value if key != 'pipeline' else type(value).__name__}")

models/churn_bundle.joblib 2,137 bytes
  pipeline         Pipeline
  features         ['orders', 'spend', 'products', 'tenure_days', 'days_since_last']
  threshold        0.07
  metrics          {'cv_auc': 0.738, 'test_auc': 0.741}
  data             3,317 customers, orders before 2011-09-01, churn = no order in next 91 days
  trained          2026-10-28
  sklearn_version  1.9.1
  mlflow_run_id    5ce8145717f540d4b2220fc6c6683e20


The pipeline is two steps: the scaler from 04.5 and the logistic regression from 03.4. Saving them **together** means nobody can forget to scale the inputs.

In [3]:
print(bundle["pipeline"])

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression(max_iter=1000))])


## 2. Reload and check the predictions are identical

A model you cannot reload exactly is not saved. Load the file into a new variable and compare its predictions with the in-memory one on 830 test customers.

In [4]:
from sklearn.model_selection import train_test_split
cust = cm.build_customer_table(cm.load_sales())
X_train, X_test, y_train, y_test = train_test_split(
    cust[cm.FEATURES], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, train {len(X_train):,} / test {len(X_test):,}")

3,317 customers, train 2,487 / test 830


In [5]:
reloaded = joblib.load(path)
p_memory = bundle["pipeline"].predict_proba(X_test)[:, 1]
p_file = reloaded["pipeline"].predict_proba(X_test)[:, 1]
print("identical predictions:", np.array_equal(p_memory, p_file))
print("max difference       :", np.abs(p_memory - p_file).max())

identical predictions: True
max difference       : 0.0


## 3. The model, by hand

The README's table, read straight out of the file. Every number in the README comes from this cell.

In [6]:
scaler, logreg = bundle["pipeline"][0], bundle["pipeline"][-1]
x = np.array([2, 310, 14, 150, 95], dtype=float)          # the 05.10 customer
z = (x - scaler.mean_) / scaler.scale_
table = pd.DataFrame({"x": x, "mean": scaler.mean_, "std": scaler.scale_, "z": z,
                      "w": logreg.coef_[0], "w*z": logreg.coef_[0] * z}, index=bundle["features"])
logit = logreg.intercept_[0] + table["w*z"].sum()
print(table.round(3))
print(f"b0 = {logreg.intercept_[0]:.3f}, logit = {logit:.3f}, p = {1 / (1 + np.exp(-logit)):.4f}")

                     x      mean       std      z      w    w*z
orders             2.0     3.371     5.189 -0.264 -0.992  0.262
spend            310.0  1510.119  5346.759 -0.224  0.060 -0.013
products          14.0    48.447    61.670 -0.559 -0.450  0.252
tenure_days      150.0   176.076    78.012 -0.334 -0.173  0.058
days_since_last   95.0    92.218    77.409  0.036  0.437  0.016
b0 = -0.476, logit = 0.098, p = 0.5244


In [7]:
new = pd.DataFrame([dict(zip(bundle["features"], x))])
print("bundle says:", cm.score(bundle, new).iloc[0].to_dict())

bundle says: {'churn_probability': 0.5244, 'voucher': True}


## 4. Seeds: why "0.741" needs a footnote

The training code fixes `random_state=42` for the train/test split. Change only that number and the test AUC moves, because a different 830 customers end up in the test set. Nothing about the model improved or got worse.

In [8]:
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
for seed in [0, 1, 7, 42, 123]:
    a, b, c, d = train_test_split(cust[cm.FEATURES], cust["churned"], test_size=0.25,
                                  random_state=seed, stratify=cust["churned"])
    m = clone(bundle["pipeline"]).fit(a, c)
    print(f"seed {seed:>3}: test AUC {roc_auc_score(d, m.predict_proba(b)[:, 1]):.3f}")

seed   0: test AUC 0.734
seed   1: test AUC 0.733
seed   7: test AUC 0.730
seed  42: test AUC 0.741
seed 123: test AUC 0.744


## 5. Versions

The bundle records the scikit-learn version it was saved with. Write the rest down too: `pip freeze` lists every installed library with its exact version. Below are the ones this model touches.

In [9]:
from importlib.metadata import version
for lib in ["scikit-learn", "numpy", "pandas", "joblib", "scipy"]:
    print(f"{lib}=={version(lib)}")

scikit-learn==1.9.1
numpy==2.5.4
pandas==3.1.0
joblib==1.6.0
scipy==1.18.1


`load_bundle` compares the stored version with the running one. Here we fake an old bundle to see the warning a teammate would get after an upgrade.

In [10]:
import warnings
old = dict(bundle, sklearn_version="1.2.0")
joblib.dump(old, ROOT / "models" / "old_bundle.joblib")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    cm.load_bundle(ROOT / "models" / "old_bundle.joblib")
print(caught[0].message)
(ROOT / "models" / "old_bundle.joblib").unlink()

Model saved with scikit-learn 1.2.0, running 1.9.1. Retrain if predictions look wrong.


## 6. Batch scoring, the way the scheduler will run it

Make a CSV of 300 customers, as retention's system would export it, and break two rows on purpose: one missing value, one negative spend.

In [11]:
batch = cust.sample(300, random_state=1).reset_index().rename(columns={"Customer ID": "customer_id"})
batch = batch[["customer_id"] + cm.FEATURES].astype({"customer_id": int})
batch.loc[3, "spend"] = np.nan
batch.loc[7, "spend"] = -50
import os
csv_in = os.path.relpath(ROOT / "models" / "customers_to_score.csv")   # a relative path, as you would type it
batch.to_csv(csv_in, index=False)
print(f"wrote {len(batch)} rows to {csv_in}")

wrote 300 rows to ../../../models/customers_to_score.csv


Now run `score_customers.py`. From a terminal this is `python score_customers.py models/customers_to_score.csv`; here we call its `main()` with the same arguments.

In [12]:
import importlib.util

def load_module(rel, name):
    """Import a .py file from the course as a module (so the notebook can test it)."""
    spec = importlib.util.spec_from_file_location(name, repo_file(rel))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

scorer = load_module("modules/10-deployment-mlops/01-saving-and-reproducibility/score_customers.py", "score_customers")
exit_code = scorer.main([csv_in])
print("exit code:", exit_code)

Skipping 2 row(s) with missing or negative values: rows [3, 7]
Scored 298 customers with the model trained 2026-10-28 (threshold 0.07): 277 get a voucher.
Written to ../../../models/customers_to_score_scored.csv
exit code: 0


In [13]:
scored = pd.read_csv(ROOT / "models" / "customers_to_score_scored.csv")
print(scored.shape)
scored.head()

(298, 8)


,customer_id,orders,spend,products,tenure_days,days_since_last,churn_probability,voucher
0,17267,2,317.20,37,27,27,0.4546,True
1,14352,4,1078.96,44,173,57,0.3185,True
2,16014,2,514.38,115,191,52,0.2747,True
3,13918,1,198.20,8,153,153,0.6575,True
4,13370,1,754.87,39,271,271,0.6978,True


In [14]:
print(f"vouchers: {scored['voucher'].sum()} of {len(scored)} ({scored['voucher'].mean():.0%})")
print(f"median churn probability: {scored['churn_probability'].median():.3f}")

vouchers: 277 of 298 (93%)
median churn probability: 0.492


A missing column is a different kind of error: the script stops with a clear message and exit code 1, so a scheduler can alert someone instead of writing an empty file.

In [15]:
bad = os.path.relpath(ROOT / "models" / "bad_export.csv")
batch.drop(columns="spend").to_csv(bad, index=False)
print("exit code:", scorer.main([bad]))

exit code: 1


ERROR: ../../../models/bad_export.csv has no column(s) ['spend']


## Quiz answers

1. Data, code, library versions and the random seed.
2. The logistic regression expects scaled inputs. Saved alone, production would feed it raw pounds and days and get wrong probabilities without any error.
3. Batch scores many rows on a schedule (the Monday CSV for retention); online scores one request as it happens (the website's risk badge, 10.2).
4. Unpickling can execute code hidden in the file. Only load files you or your team created.
5. The scheduler used a different Python (not the course environment), so the libraries are missing. Use the full path to `.venv`'s Python.

**Next:** [10.2 A prediction API with FastAPI](../02-fastapi/)